<a href="https://colab.research.google.com/github/Ed-Neema/Financial-Sentiment-Classification/blob/main/Finance_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Finetuning DistilBert**

In [3]:
pip install --upgrade transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [4]:
import torch
print(torch.cuda.is_available())  # should print True

True


In [5]:
from datasets import load_dataset

# Load the highest-agreement configuration
from datasets import load_dataset

ds = load_dataset("atrost/financial_phrasebank")
print(ds)
print(ds["train"][0])


README.md:   0%|          | 0.00/721 [00:00<?, ?B/s]

data/train-00000-of-00001-138b53eb17a3e8(…): reconstructing file:   0%|          |  0.00B /  268kB            

data/train-00000-of-00001-138b53eb17a3e8(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-0876be41e(…): reconstructing file:   0%|          |  0.00B / 68.7kB            

data/validation-00000-of-00001-0876be41e(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-41c7ea948573445(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/776 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 776
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 970
    })
})
{'sentence': 'EBIT margin was up from 1.4 % to 5.1 % .', 'label': 2}


- Must use the exact tokenizer that matches your model.

- DistilBERT learned English using one specific vocabulary — a fixed dictionary where every known word-piece has an assigned ID number. If you fed it text numbered by a different scheme, the numbers would mean nothing to it, like handing someone a book written in an alphabet they've never seen.

- from_pretrained("distilbert/distilbert-base-uncased") downloads that model's own matching vocabulary so the numbers line up with what it learned.

In [ ]:
"""
"""
from transformers import AutoTokenizer
#AutoTokenizer is a convenient class that automatically selects the correct tokenizer for a given pre-trained model.

"""
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased"):
This initializes a tokenizer specifically for the distilbert-base-uncased model. from_pretrained() downloads the pre-trained tokenizer's vocabulary and configuration, ensuring that the tokenization process is consistent with how the model was trained.
'Uncased' means it will convert all text to lowercase.
"""
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

#This defines a function named preprocess_function.
#It takes a dictionary of examples (typically a batch from the dataset) and applies the tokenizer to the "sentence" key within those examples. truncation=True ensures that if a sentence is longer than the tokenizer's maximum input length, it will be cut off to fit, preventing errors.
#each item looks like: {'sentence': 'EBIT margin was up from 1.4 % to 5.1 % .', 'label': 2}
def preprocess_function(examples):
    return tokenizer(examples["sentence"], truncation=True)
tokenized = ds.map(preprocess_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3100 [00:00<?, ? examples/s]

Map:   0%|          | 0/776 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

In [ ]:
"""
The collator needs the tokenizer for two reasons:
- to know which token ID means "padding" (each model reserves a specific number for filler),
- to know which side to pad on. So DataCollatorWithPadding(tokenizer=tokenizer) means "make me a padding machine set up for DistilBERT's rules."
"""
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


- import numpy as np: This line imports the numpy library, which is a fundamental package for numerical computation in Python. It's often used for array manipulations and mathematical operations, which will be useful here for processing predictions.
- import evaluate: This imports the evaluate library, which provides a simple way to load and compute various evaluation metrics commonly used in machine learning.
- accuracy = evaluate.load("accuracy"): This line loads the accuracy metric from the evaluate library. Once loaded, accuracy becomes an object that can be called later to compute the accuracy score.
- def compute_metrics(eval_pred):: This defines a function called compute_metrics. This function is typically passed to a Trainer in the transformers library and is responsible for calculating metrics during evaluation.
- predictions, labels = eval_pred: The eval_pred argument, when passed from a Trainer, is a tuple containing the raw model predictions (logits) and the true labels. This line unpacks those two components into separate variables.
- predictions = np.argmax(predictions, axis=1): Model predictions (logits) are usually raw scores for each class. To get the actual predicted class, you need to find the index of the highest score for each example. np.argmax(predictions, axis=1) does exactly that, converting the raw predictions into a single predicted class ID for each sample.
- return accuracy.compute(predictions=predictions, references=labels): Finally, this line computes the accuracy. It calls the compute method of the accuracy object (which we loaded earlier), passing in the processed predictions (the predicted class IDs) and the labels (the true class IDs or references). The function then returns the calculated accuracy score.

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id="Ed-Akoth/financial-sentiment-distilbert",

)

In [ ]:
from huggingface_hub import login
login()  # paste an access token when prompted (from huggingface.co/settings/tokens)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.408110,0.837629
2,No log,0.409200,0.846649
3,0.390930,0.448764,0.846649
4,0.390930,0.564243,0.847938
5,0.390930,0.595608,0.846649


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=970, training_loss=0.2557056702289385, metrics={'train_runtime': 235.5461, 'train_samples_per_second': 65.805, 'train_steps_per_second': 4.118, 'total_flos': 238732242685416.0, 'train_loss': 0.2557056702289385, 'epoch': 5.0})

In [ ]:
test_results = trainer.evaluate(tokenized["test"])
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy
0.390930,0.431080,5,0.828866


{'eval_loss': 0.43107980489730835, 'eval_accuracy': 0.8288659793814434}


# **Finetuning FinBert**

##1.  Fine-tune FinBERT

In [7]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
import numpy as np
import evaluate

# Dataset (reuse — already loaded, but shown for completeness)
ds = load_dataset("atrost/financial_phrasebank")

# Your label scheme
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

# Tokenizer — FinBERT's own
finbert_tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

def preprocess_function(examples):
    return finbert_tokenizer(examples["sentence"], truncation=True)

tokenized = ds.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=finbert_tokenizer)

# Fresh classification head, sized to your labels
finbert_finetuned = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)
#####-----run 1
# training_args = TrainingArguments(
#     output_dir="./finbert-results",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,          # dropped from 5 — accuracy plateaued early last time
#     weight_decay=0.01,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     push_to_hub=False,
# )

#####-----run 2
"""
Lowering Learning Rate:
-  FinBERT already has valuable financial-domain knowledge baked into its base from pretraining
-  learning rate that's too high can partially overwrite that knowledge during fine-tuning —
a phenomenon called catastrophic forgetting.
- DistilBERT (general English) has less domain-specific knowledge to protect, so a higher rate was safer for it;
- FinBERT benefits from being nudged more gently.
"""
training_args = TrainingArguments(
    output_dir="./finbert-results",
    learning_rate=1e-5, # down from 2e-5
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,          # dropped from 5 — accuracy plateaued early last time
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)


trainer = Trainer(
    model=finbert_finetuned,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=finbert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# The number that matters
finbert_finetuned_results = trainer.evaluate(tokenized["test"])
print("Fine-tuned FinBERT test results:", finbert_finetuned_results)

Map:   0%|          | 0/3100 [00:00<?, ? examples/s]

Map:   0%|          | 0/776 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.349017,0.873711
2,No log,0.337104,0.878866
3,0.349260,0.373639,0.875000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.349260,0.419386,3,0.853608


Fine-tuned FinBERT test results: {'eval_loss': 0.41938599944114685, 'eval_accuracy': 0.8536082474226804}


##2. Test the original authors' classifier head (no training at all)

In [11]:
import torch
from torch.utils.data import DataLoader

# Load FinBERT exactly as published — original head intact
finbert_original = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

print("FinBERT's own label scheme:", finbert_original.config.id2label)  # confirm before trusting it

# Tokenize test set (same tokenizer, already fine since it's the same base model)
test_tokenized = tokenized["test"]
test_for_loader = tokenized["test"].remove_columns(["sentence"])

loader = DataLoader(test_for_loader, batch_size=16, collate_fn=data_collator)

# loader = DataLoader(test_tokenized, batch_size=16, collate_fn=data_collator)

your_id2label = {0: "negative", 1: "neutral", 2: "positive"}  # your dataset's scheme

finbert_original.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in loader:
        true_labels = batch.pop("labels")
        outputs = finbert_original(**batch)
        preds = outputs.logits.argmax(dim=-1)

        for pred_id, true_id in zip(preds, true_labels):
            pred_label = finbert_original.config.id2label[pred_id.item()]  # FinBERT's own naming
            true_label = your_id2label[true_id.item()]                     # your dataset's naming
            if pred_label == true_label:
                correct += 1
            total += 1

finbert_original_accuracy = correct / total
print(f"Off-the-shelf FinBERT accuracy: {finbert_original_accuracy:.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT's own label scheme: {0: 'positive', 1: 'negative', 2: 'neutral'}
Off-the-shelf FinBERT accuracy: 0.8649


# The Comparison

In [12]:
print("=== Model Comparison on Test Set ===")
print(f"DistilBERT (fine-tuned):     0.8289")  # your earlier result
print(f"FinBERT (fine-tuned):        {finbert_finetuned_results['eval_accuracy']:.4f}")
print(f"FinBERT (off-the-shelf):     {finbert_original_accuracy:.4f}")

=== Model Comparison on Test Set ===
DistilBERT (fine-tuned):     0.8289
FinBERT (fine-tuned):        0.8536
FinBERT (off-the-shelf):     0.8649


# **Finetuning FinBERT and Using Optuna for Hyperparameter tuning**

In [2]:
# --------------------------------------------------------------------------
# SECTION 0 — Install Optuna
# --------------------------------------------------------------------------
# Optuna isn't included in Colab by default, so we install it once here.
# This only needs to run once per Colab session.
#Remember to run the installs in the first cell of this colab notebook
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.9 MB/s eta 0:00:00


In [4]:
# --------------------------------------------------------------------------
# SECTION 1 — Imports
# --------------------------------------------------------------------------
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
import evaluate

In [5]:
# --------------------------------------------------------------------------
# SECTION 2 — Load the dataset
# --------------------------------------------------------------------------
# This dataset already comes pre-split into train / validation / test,
# so we don't need to make our own split like we did with IMDb.
ds = load_dataset("atrost/financial_phrasebank")

# Quick sanity check
print(ds)
print(ds["train"][0])

README.md:   0%|          | 0.00/721 [00:00<?, ?B/s]

data/train-00000-of-00001-138b53eb17a3e8(…): reconstructing file:   0%|          |  0.00B /  268kB            

data/train-00000-of-00001-138b53eb17a3e8(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-0876be41e(…): reconstructing file:   0%|          |  0.00B / 68.7kB            

data/validation-00000-of-00001-0876be41e(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-41c7ea948573445(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/776 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3100
    })
    validation: Dataset({
        features: ['sentence', 'label'],
        num_rows: 776
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 970
    })
})
{'sentence': 'EBIT margin was up from 1.4 % to 5.1 % .', 'label': 2}


In [6]:
# --------------------------------------------------------------------------
# SECTION 3 — Define the label scheme
# --------------------------------------------------------------------------
# These dictionaries tell the model how to translate between the numeric
# labels in the dataset (0, 1, 2) and human-readable names. We define this
# ourselves because we're training a fresh classification head — not using
# whatever label order the original FinBERT authors happened to use.
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

In [7]:

# --------------------------------------------------------------------------
# SECTION 4 — Tokenizer and preprocessing
# --------------------------------------------------------------------------
# We load FinBERT's own tokenizer — every model has its own vocabulary,
# so we must use the tokenizer that matches the model we're fine-tuning.
finbert_tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

def preprocess_function(examples):
    # Turns raw sentences into token IDs the model can read.
    # truncation=True cuts off anything longer than the model's max length,
    # so a very long sentence can't crash the model.
    return finbert_tokenizer(examples["sentence"], truncation=True)

# .map() applies preprocess_function to every row in every split
# (train/validation/test) at once. batched=True speeds this up by
# processing many rows together instead of one at a time.
tokenized = ds.map(preprocess_function, batched=True)

# The data collator pads each batch of examples to the length of the
# longest example *in that batch* (not the whole dataset), which is more
# efficient than padding everything to one fixed maximum length.
data_collator = DataCollatorWithPadding(tokenizer=finbert_tokenizer)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/3100 [00:00<?, ? examples/s]

Map:   0%|          | 0/776 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

In [8]:
# --------------------------------------------------------------------------
# SECTION 5 — model_init function (required for Optuna search)
# --------------------------------------------------------------------------
# Normally you'd create the model once. But Optuna needs to build a BRAND
# NEW, untrained model for every trial it runs — otherwise trial 2 would
# start from trial 1's already-trained weights, which would invalidate
# the comparison between trials. Wrapping model creation in a function lets
# Trainer call it fresh each time.
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        "ProsusAI/finbert",
        num_labels=3,                  # 3 classes: negative/neutral/positive
        id2label=id2label,
        label2id=label2id,
        # FinBERT ships with its own pretrained classification head, sized
        # and ordered for its own label scheme. We discard that head here
        # and attach a fresh, untrained one that matches OUR label scheme,
        # so the whole comparison is fair and fully our own work.
        ignore_mismatched_sizes=True,
    )


In [9]:
# --------------------------------------------------------------------------
# SECTION 6 — Evaluation metric
# --------------------------------------------------------------------------
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    # eval_pred contains the model's raw output scores (predictions) and
    # the true labels. We take the highest-scoring class as the model's
    # actual prediction, then compare it against the true labels.
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [10]:
# --------------------------------------------------------------------------
# SECTION 7 — Base training arguments (used during the search phase)
# --------------------------------------------------------------------------
# These are placeholder/default values. During the Optuna search, most of
# these will be OVERRIDDEN per-trial by the search space defined below —
# this object just fills in anything the search space doesn't touch
# (e.g. eval_strategy, output_dir).
search_args = TrainingArguments(
    output_dir="./finbert-search",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    per_device_eval_batch_size=16,
    push_to_hub=False,
    # Keeping this quiet-ish; Optuna will run several full training loops
    # back to back, and we don't need every log line for every trial.
    report_to="none",
)

In [11]:
# --------------------------------------------------------------------------
# SECTION 8 — Trainer, set up for hyperparameter search
# --------------------------------------------------------------------------
# Note: model_init instead of model — this is what makes Optuna search
# possible (see Section 5's explanation).
trainer = Trainer(
    model_init=model_init,
    args=search_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],   # validation, not test — test stays untouched
    processing_class=finbert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [12]:
# --------------------------------------------------------------------------
# SECTION 9 — Define the Optuna search space
# --------------------------------------------------------------------------
# This function tells Optuna what values it's allowed to try, and for
# which hyperparameters. Optuna calls this once per trial, and "trial"
# is Optuna's object for suggesting a value within your specified range.
def optuna_hp_space(trial):
    return {
        # log=True means Optuna searches this range on a logarithmic
        # scale rather than a linear one — appropriate for learning rates,
        # since 1e-5 -> 2e-5 is proportionally similar to 2e-5 -> 4e-5.
        "learning_rate": trial.suggest_float("learning_rate", 5e-6, 5e-5, log=True),

        # Capped at 3 (not 5) because we already saw accuracy plateau by
        # epoch 2 in earlier runs — no point letting trials waste time
        # training epochs that likely won't help.
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 3),

        # suggest_categorical is for picking from a fixed set of options,
        # rather than any number in a range — batch size only makes sense
        # at specific values.
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [8, 16, 32]
        ),

        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }


In [13]:
# --------------------------------------------------------------------------
# SECTION 10 — Define what "better" means
# --------------------------------------------------------------------------
# We want Optuna to search for hyperparameters that MAXIMIZE accuracy
# (as opposed to minimizing loss, which is a different, related goal).
"""
# Here I was worried about not being able to detect when my model is overfitting because accuracy here is what's being maximized
# But there are already existing guards for this:

1. eval_strategy="epoch",
save_strategy="epoch",
load_best_model_at_end=True,
fter every epoch within a trial, the model gets evaluated on the validation set, and Trainer remembers which epoch's checkpoint scored best — by default,
this means lowest validation loss, not accuracy. At the end of that trial, load_best_model_at_end=True throws away the later, more-overfit epochs and keeps the best one.

2.eval_accuracy in that function is validation accuracy.
It's the accuracy computed on tokenized["validation"] — data the model isn't trained on.
So Optuna isn't rewarding a trial for memorizing the training set;
it's rewarding a trial for generalizing well to unseen validation data.
"""
def compute_objective(metrics):
    return metrics["eval_accuracy"]


In [14]:
# --------------------------------------------------------------------------
# SECTION 11 — Run the search
# --------------------------------------------------------------------------
# This runs n_trials complete training runs, each with a different
# hyperparameter combination suggested by Optuna. After each trial,
# Optuna uses the result to make a smarter guess for the next one.
#
# n_trials=5 is a deliberately modest number to respect Colab's free GPU
# session limits — each trial is a full training run, so this could take
# a while. Increase later if you have GPU time to spare.
best_trial = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    compute_objective=compute_objective,
    n_trials=5,
)

[I 2026-07-20 10:20:57,472] A new study created in memory with name: no-name-8f3e9d9c-b417-465a-92cf-732641873598


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.377479,0.853093
2,0.645076,0.306665,0.872423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.452040,0.833763
2,No log,0.377064,0.854381


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-07-20 10:33:26,590] Trial 4 finished with value: 0.854381443298969 and parameters: {'learning_rate': 2.3039620128143535e-05, 'num_train_epochs': 2, 'per_device_train_batch_size': 32, 'weight_decay': 0.07888093338583607}. Best is trial 0 with value: 0.8724226804123711.


In [15]:

# Shows the winning hyperparameter combination and the accuracy it reached.
print("Best trial found by Optuna:")
print(best_trial)


Best trial found by Optuna:
BestRun(run_id='0', objective=0.8724226804123711, hyperparameters={'learning_rate': 7.357226650046562e-06, 'num_train_epochs': 2, 'per_device_train_batch_size': 8, 'weight_decay': 0.010286414963852865}, run_summary=None)


In [16]:
# --------------------------------------------------------------------------
# SECTION 12 — Retrain using the best hyperparameters found
# --------------------------------------------------------------------------
# The search above only tells us the best SETTINGS — it doesn't leave us
# a saved, usable model (Optuna trials aren't kept). So we now do one
# final, real training run using those winning settings.
final_args = TrainingArguments(
    output_dir="./finbert-final",
    learning_rate=best_trial.hyperparameters["learning_rate"],
    num_train_epochs=best_trial.hyperparameters["num_train_epochs"],
    per_device_train_batch_size=best_trial.hyperparameters["per_device_train_batch_size"],
    weight_decay=best_trial.hyperparameters["weight_decay"],
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)
final_trainer = Trainer(
    model_init=model_init,
    args=final_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=finbert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

final_trainer.train()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.377479,0.853093
2,0.645076,0.306665,0.872423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=776, training_loss=0.51320753392485, metrics={'train_runtime': 137.9472, 'train_samples_per_second': 44.945, 'train_steps_per_second': 5.625, 'total_flos': 168877702709856.0, 'train_loss': 0.51320753392485, 'epoch': 2.0})

In [17]:
final_trainer = Trainer(
    model_init=model_init,
    args=final_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=finbert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

final_trainer.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.377479,0.853093
2,0.645076,0.306665,0.872423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=776, training_loss=0.51320753392485, metrics={'train_runtime': 123.2184, 'train_samples_per_second': 50.317, 'train_steps_per_second': 6.298, 'total_flos': 168877702709856.0, 'train_loss': 0.51320753392485, 'epoch': 2.0})

In [18]:
# --------------------------------------------------------------------------
# SECTION 13 — Final, honest evaluation on the untouched test set
# --------------------------------------------------------------------------
# This is the number that goes in your README. The test set was never
# used during training or during the Optuna search, so this accuracy
# reflects genuine, unbiased performance on unseen data.
final_test_results = final_trainer.evaluate(tokenized["test"])
print("Final tuned FinBERT test results:", final_test_results)


Training Loss,Validation Loss,Epoch,Accuracy
0.645076,0.387951,2,0.860825


Final tuned FinBERT test results: {'eval_loss': 0.3879507780075073, 'eval_accuracy': 0.8608247422680413}


In [19]:
# --------------------------------------------------------------------------
# SECTION 14 — Save the model locally
# --------------------------------------------------------------------------
# This saves the model weights, config, and tokenizer together into one
# folder, so it can be loaded later with just the folder path.
final_trainer.save_model("financial-sentiment-model")
finbert_tokenizer.save_pretrained("financial-sentiment-model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('financial-sentiment-model/tokenizer_config.json',
 'financial-sentiment-model/tokenizer.json')

In [19]:
# --------------------------------------------------------------------------
# SECTION 15 — Zip it up for download
# --------------------------------------------------------------------------
# Colab notebooks run on a remote machine, so we zip the model folder
# and download it locally, then move it into services/sentiment-service/model/
# in your project.
!zip -r financial-sentiment-model.zip financial-sentiment-model

	zip warning: name not matched: financial-sentiment-model

zip error: Nothing to do! (try: zip -r financial-sentiment-model.zip . -i financial-sentiment-model)


In [24]:
!ls -la finbert-final

total 16
drwxr-xr-x 4 root root 4096 Jul 20 10:35 .
drwxr-xr-x 1 root root 4096 Jul 20 10:33 ..
drwxr-xr-x 2 root root 4096 Jul 20 10:36 checkpoint-388
drwxr-xr-x 2 root root 4096 Jul 20 10:37 checkpoint-776


In [25]:
final_trainer.save_model("financial-sentiment-model")
finbert_tokenizer.save_pretrained("financial-sentiment-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('financial-sentiment-model/tokenizer_config.json',
 'financial-sentiment-model/tokenizer.json')

In [26]:
!ls -la financial-sentiment-model

total 428424
drwxr-xr-x 2 root root      4096 Jul 20 10:48 .
drwxr-xr-x 1 root root      4096 Jul 20 10:48 ..
-rw-r--r-- 1 root root       973 Jul 20 10:48 config.json
-rw------- 1 root root 437961724 Jul 20 10:48 model.safetensors
-rw-r--r-- 1 root root       351 Jul 20 10:48 tokenizer_config.json
-rw-r--r-- 1 root root    711494 Jul 20 10:48 tokenizer.json
-rw-r--r-- 1 root root      5201 Jul 20 10:48 training_args.bin


In [27]:
#Then — before zipping — back it up to Drive immediately, so a disconnect from here on can't cost you anything
from google.colab import drive
drive.mount('/content/drive')
!cp -r financial-sentiment-model /content/drive/MyDrive/financial-sentiment-model

Mounted at /content/drive


In [28]:
!zip -r financial-sentiment-model.zip financial-sentiment-model
from google.colab import files
files.download("financial-sentiment-model.zip")

  adding: financial-sentiment-model/ (stored 0%)
  adding: financial-sentiment-model/tokenizer_config.json (deflated 43%)
  adding: financial-sentiment-model/model.safetensors (deflated 7%)
  adding: financial-sentiment-model/tokenizer.json (deflated 71%)
  adding: financial-sentiment-model/config.json (deflated 54%)
  adding: financial-sentiment-model/training_args.bin (deflated 54%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>